# cd_02_extract_outbuild

Pulls every Outbuild endpoint in `Files/config/outbuild_endpoints.yml` into
**CD_Bronze_Lakehouse**. Outbuild is the ONLY milestone source (fct_Milestone, Schedule page).

No endpoint logic here: requests, paging, transient-5xx retry, the bronze row shape, raw
archive, manifest and the failure policy live in `extract_outbuild_local.py` (`extract()`).
Every endpoint is a full pull, so there are no watermarks.

Failure policy: an endpoint that still fails after retries fails this notebook only if it is
`consumed: true` (read by silver: projects, activities). Any other failure is a warning in
`Files/_diag/ingestion/<batch>.json`.

Generated by `_local/make_notebooks.py`; deployed by `_local/deploy_outbuild.py`.

In [ ]:
import sys
sys.path.insert(0, "/lakehouse/default/Files/lib")

from pyspark.sql import functions as F

import fabric_common as fc
import extract_outbuild_local as ob

CONFIG = "/lakehouse/default/Files/config/outbuild_endpoints.yml"
DIAG = "/lakehouse/default/Files/_diag"
# The contract cd_05_land_to_bronze writes; the existing tables were created by it.
COLUMNS = ["_key", "_project_id", "payload", "_ingested_at", "_batch_id",
           "_row_hash", "_source_endpoint", "_merge_key"]

batch_id = fc.new_batch_id()
endpoints = ob.load_registry(CONFIG)
# Fails closed inside Fabric: a missing secret raises here, before any request.
token = fc.get_secret("OUTBUILD_API_TOKEN")
print(f"batch {batch_id}: {len(endpoints)} endpoints")

In [ ]:
def write(table, rows):
    df = spark.createDataFrame([[r[c] for c in COLUMNS] for r in rows],
                               ", ".join(f"`{c}` string" for c in COLUMNS))
    df = fc.prepare_merge(df.withColumn("_ingested_at", F.to_timestamp("_ingested_at")),
                          ["_merge_key"])
    if not spark.catalog.tableExists(table):
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    else:
        df.createOrReplaceTempView("_outbuild_staged")
        spark.sql(fc.merge_sql(table, "_outbuild_staged", ["_merge_key"], COLUMNS))
    n = df.count()
    fc.log_run(spark, batch_id, "extract_outbuild", table, n)
    return n

manifest = ob.extract(endpoints, token, batch_id, DIAG, write)

for a in manifest["endpoints"]:
    detail = a.get("error") or a.get("note") or ""
    print(f"  {a['endpoint']:<28} {a['status']:<9} {a['written_rows']:>7} rows  {detail[:160]}")
print(f"\n{manifest['total_rows']} rows, status {manifest['status']}")
if manifest["warnings"]:
    print(f"WARNING - unconsumed endpoint(s) failed, not blocking: {', '.join(manifest['warnings'])}")
if manifest["blocking_failures"]:
    raise RuntimeError("consumed Outbuild endpoint(s) failed: "
                       + ", ".join(manifest["blocking_failures"])
                       + f" - see Files/_diag/ingestion/{batch_id}.json")